#1. Read Silver Tables

We read the cleaned flight, airline and airport data from Silver.
We also check the number of records in each table.

In [0]:
from pyspark.sql import functions as F

SCHEMA = "flight_delay"
SILVER_FLIGHTS = f"{SCHEMA}.silver_flights"
SILVER_AIRLINES = f"{SCHEMA}.silver_airlines"
SILVER_AIRPORTS = f"{SCHEMA}.silver_airports"
flights = spark.table(SILVER_FLIGHTS)
airlines = spark.table(SILVER_AIRLINES)
airports = spark.table(SILVER_AIRPORTS)
print(f"Flights: {flights.count():,}")
print(f"Airlines: {airlines.count():,}")
print(f"Airports: {airports.count():,}")

# 2. Preparing data for BI

We create two new columns:

- `FLIGHT_DATE` — the full date of the flight.
- `IS_DELAYED` — 1 if the departure delay is at least 15 minutes, otherwise 0.

If the departure delay is unknown, `IS_DELAYED` remains **NULL**.
We keep all flight records.

In [0]:
flights_gold = (flights.withColumn(
    "FLIGHT_DATE",
    F.make_date(F.col("YEAR"), F.col("MONTH"), F.col("DAY"))
  )
  .withColumn("IS_DELAYED",
              F.when(F.col("DEPARTURE_DELAY").isNull(), None)
              .when(F.col("DEPARTURE_DELAY") >= 15, 1)
              .otherwise(0)))

flights_gold.select("FLIGHT_DATE", "DEPARTURE_DELAY", "IS_DELAYED", "CANCELLED", "DIVERTED").show(10)

# 3. Join with airlines and airports

Preparing lookup tables with supported fields.
This gives us airline names, airport names, cities, and states.
**LEFT JOIN** keeps flights even if some lookup information is missing.
We also check that the number of flights did not change after the joins.

In [0]:
airlines_lookup = airlines.select(F.col("IATA_CODE").alias("AIRLINE_CODE"),
    F.col("AIRLINE").alias("AIRLINE_NAME")
)

airports_lookup = airports.select(F.col("IATA_CODE").alias("AIRPORT_CODE"),
    F.col("AIRPORT").alias("AIRPORT_NAME"),
    F.col("STATE"), F.col("CITY"))

In [0]:
flights_gold = (flights_gold.join(airlines_lookup,
    F.col("AIRLINE") == F.col("AIRLINE_CODE"), "left").drop("AIRLINE_CODE"))

flights_gold = (flights_gold.join(airports_lookup.select(
    F.col("AIRPORT_CODE").alias("ORIGIN_CODE"),
    F.col("AIRPORT_NAME").alias("ORIGIN_NAME"),
    F.col("CITY").alias("ORIGIN_CITY_TOWN"), #with no name
    F.col("STATE").alias("ORIGIN_STATE")
), F.col("ORIGIN_AIRPORT") == F.col("ORIGIN_CODE"), "left").drop("ORIGIN_CODE"))

flights_gold = (flights_gold.join(airports_lookup.select(
    F.col("AIRPORT_CODE").alias("DEST_CODE"),
    F.col("AIRPORT_NAME").alias("DEST_NAME"),
    F.col("CITY").alias("DEST_CITY_TOWN"),
    F.col("STATE").alias("DEST_STATE")
), F.col("DESTINATION_AIRPORT") == F.col("DEST_CODE"), "left").drop("DEST_CODE"))

flights_gold.select("AIRLINE", "AIRLINE_NAME", "ORIGIN_AIRPORT",
    "ORIGIN_NAME", "DESTINATION_AIRPORT", "DEST_NAME").show(5,
     truncate=False)

In [0]:
print(f"Silver flights: {flights.count():,}")
print(f"Gold flights: {flights_gold.count():,}")

assert flights.count() == flights_gold.count(), (
    "Row count changed after JOIN!"
)

# 4. Create KPIs for BI Dashboard

We calculate flight statistics for the whole dataset, each airline and each departure airport.
The KPIs include total flights, delayed flights, cancelled flights, diverted flights and delay rate.
Delay rate is calculated only for flights with known departure delays.

In [0]:
flight_kpis = flights_gold.agg(
    F.count("*").alias("TOTAL_FLIGHTS"),
    F.sum("IS_DELAYED").alias("DELAYED_FLIGHTS"),
    F.sum("CANCELLED").alias("CANCELLED_FLIGHTS"),
    F.sum("DIVERTED").alias("DIVERTED_FLIGHTS"),
    F.round(F.avg("IS_DELAYED") * 100, 2).alias("DELAY_RATE")
)

flight_kpis.show()

In [0]:
airline_kpis = (flights_gold.groupBy("AIRLINE", "AIRLINE_NAME").agg(
    F.count("*").alias("TOTAL_FLIGHTS"),
    F.sum("IS_DELAYED").alias("DELAYED_FLIGHTS"),
    F.sum("CANCELLED").alias("CANCELLED_FLIGHTS"),
    F.sum("DIVERTED").alias("DIVERTED_FLIGHTS"),
    F.round(F.avg("IS_DELAYED") * 100, 2).alias("DELAY_RATE")
).orderBy(F.col("DELAY_RATE").desc()))

airline_kpis.show(20, truncate=False)

In [0]:
airport_kpis = (flights_gold.groupBy(
    "ORIGIN_AIRPORT", "ORIGIN_NAME",
    "ORIGIN_CITY_TOWN", "ORIGIN_STATE"
).agg(F.count("*").alias("TOTAL_FLIGHTS"), F.sum("IS_DELAYED").alias("DELAYED_FLIGHTS"),
      F.sum("CANCELLED").alias("CANCELLED_FLIGHTS"),
      F.sum("DIVERTED").alias("DIVERTED_FLIGHTS"),
      F.round(F.avg("IS_DELAYED")*100, 2).alias("DELAY_RATE"))
      .orderBy(F.col("DELAY_RATE").desc()))

airport_kpis.show(20, truncate=False)

**Daily KPIs**: Building the table showing the daily changes in flight delays, cancellations and diversions.

In [0]:
daily_kpis = (flights_gold.groupBy("FLIGHT_DATE").agg(
    F.count("*").alias("TOTAL_FLIGHTS"),
    F.sum("IS_DELAYED").alias("DELAYED_FLIGHTS"),
    F.sum("CANCELLED").alias("CANCELLED_FLIGHTS"),
    F.sum("DIVERTED").alias("DIVERTED_FLIGHTS"),
    F.round(F.avg("IS_DELAYED")*100, 2).alias("DELAY_RATE")
).orderBy("FLIGHT_DATE"))

daily_kpis.show(20, truncate=False)

# 5. Save Gold Tables

Saving Data Frames `flights_gold`, `flight_kpis`, `airline_kpis`, `airport_kpis`, `daily_kpis` in Delta Tables in order to use them later in AI/BI Dashboards.

In [0]:
GOLD_FLIGHTS = f"{SCHEMA}.gold_flights"
GOLD_FLIGHT_KPIS = f"{SCHEMA}.gold_flight_kpis"
GOLD_AIRLINE_KPIS = f"{SCHEMA}.gold_airline_kpis"
GOLD_AIRPORT_KPIS = f"{SCHEMA}.gold_airport_kpis"
GOLD_DAILY_KPIS = f"{SCHEMA}.gold_daily_kpis"

flights_gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_FLIGHTS)
print(f"Saved: {GOLD_FLIGHTS}")
flight_kpis.write.format("delta").mode("overwrite").saveAsTable(GOLD_FLIGHT_KPIS)
print(f"Saved: {GOLD_FLIGHT_KPIS}")
airline_kpis.write.format("delta").mode("overwrite").saveAsTable(GOLD_AIRLINE_KPIS)
print(f"Saved: {GOLD_AIRLINE_KPIS}")
airport_kpis.write.format("delta").mode("overwrite").saveAsTable(GOLD_AIRPORT_KPIS)
print(f"Saved: {GOLD_AIRPORT_KPIS}")
daily_kpis.write.format("delta").mode("overwrite").saveAsTable(GOLD_DAILY_KPIS)
print(f"Saved: {GOLD_DAILY_KPIS}")

# 6. Validate Gold Tables

Ensure these were saved in Delta throughoutly.

In [0]:
flights_check = spark.table(GOLD_FLIGHTS)
flight_kpis_check = spark.table(GOLD_FLIGHT_KPIS)
airline_kpis_check = spark.table(GOLD_AIRLINE_KPIS)
airport_kpis_check = spark.table(GOLD_AIRPORT_KPIS)
daily_kpis_check = spark.table(GOLD_DAILY_KPIS)

assert flights_check.count() == flights_gold.count()
assert flight_kpis_check.count() == flight_kpis.count()
assert airline_kpis_check.count() == airline_kpis.count()
assert airport_kpis_check.count() == airport_kpis.count()
assert daily_kpis_check.count() == daily_kpis.count()

print("All of the Gold Tables were saved successfully")

And show them just to be sure

In [0]:
flight_kpis_check.show()
airline_kpis_check.show(5, truncate=False)
airport_kpis_check.show(5, truncate=False)
daily_kpis_check.show(5)


# 7. Data Quality Checks

**1) Check that joins didnt't change amount of flights** because filtering was led on Silver-level

In [0]:
assert flights_check.count() == flights.count(), "Flights amount changed after joining"

**2) Check that airline total matches overall total**

In [0]:
total_flights = flight_kpis_check.first()["TOTAL_FLIGHTS"]
airline_total = airline_kpis_check.agg(F.sum("TOTAL_FLIGHTS")).first()[0]
assert airline_total == total_flights, "Airline Totals Different"

**3) Now the same for daily flight stats**

In [0]:
daily_total = daily_kpis_check.agg(F.sum("TOTAL_FLIGHTS")).first()[0]
assert daily_total == total_flights, "Daily Totals Different"

**4) Check of delay rate value rationality**

In [0]:
invalid_rates = airline_kpis_check.filter((F.col("DELAY_RATE") < 0) |
    (F.col("DELAY_RATE") > 100)).count()
assert invalid_rates == 0, "Invalid Delay Rates Exist"